# Indexing SciDocs by OpenSearch for BM25 Model

- [beir/scidocs](https://ir-datasets.com/beir.html#beir/scidocs)

### Install python modules

In [1]:
import sys
!{sys.executable} -m pip install ir_datasets pandas opensearch-py

### Load helper modules

In [2]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

- [Local Testing Mode](install.md)

In [3]:
from opensearchpy import OpenSearch

In [4]:
host = 'localhost'
port = 9200

client = OpenSearch(
    hosts = [{'host': host, 'port': port}],
    http_compress = True,
    use_ssl = False,
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)

In [5]:
pprint.pprint(client.info())

### Index a Corpus for BM25 Model

In [6]:
import ir_datasets
dataset_name = "beir/scidocs"
dataset = ir_datasets.load(dataset_name)

Index structure

In [8]:
index_name = "scidocs_bm25"
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}
response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

Indexing

In [10]:
for doc in tqdm(dataset.docs_iter(), desc="Indexing"):
    doc_body = {
        "docid": doc.doc_id,
        "title": doc.title,
        "text": doc.text
    }
    response = client.index(index=index_name, body=doc_body)

#### Search Test

In [11]:
def search(query: str, size: int = 10) -> dict:
    body = {
        "size": size,
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["title^2", "text"] # title gets a boost
            }
        },
    }

    return client.search(index=index_name, body=body)

In [12]:
q = "Ad Hoc Retrieval Experiments Using WordNet"
resp = search(q, size=5)

print(f"\nTop {len(resp['hits']['hits'])} hits for query: {q}\n")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")